# AnomalyMatch pipeline validation on GalaxyMNIST (GPU)

**Why this notebook exists:** this is Setup Step 1 of the `image_anomaly_detection` project in `chandra-toolkit` - before adapting AnomalyMatch to real Chandra X-ray imaging cutouts, we need to confirm the pipeline itself (data loading, FixMatch training, active-learning correction cycles, evaluation) actually works, using a small public benchmark the method's own paper already validated against.

It was first run locally on a CPU-only Windows laptop. One full training cycle completed successfully there, but a *second* training cycle in the same process reliably crashed with a native access violation deep inside PyTorch's CPU convolution/activation kernels (a different operation each time - `conv2d`, then `hardtanh` - which is the signature of memory corruption in the CPU build, not a fixable bug in any one op). Both standard remedies (single-threaded MKL/OpenMP, disabling MKL-DNN's optimized CPU conv path) failed to resolve it. Since both crashing code paths are CPU-kernel-specific, GPU execution (a completely different code path via cuDNN) should sidestep the issue entirely - hence this notebook.

**Before running:** in the Kaggle notebook settings panel (right sidebar), set **Accelerator -> GPU T4 x2** (or P100). This notebook will not benefit from - and does not need - a TPU. Everything below runs without any local Chandra-specific code; it's a clean reproduction of the method's own published benchmark.

## What method is being tested: AnomalyMatch

[AnomalyMatch](https://github.com/esa/AnomalyMatch) (Gomez et al., [arXiv:2505.03509](https://arxiv.org/abs/2505.03509), ESA) is a semi-supervised anomaly-detection method for astronomical images, built from three pieces:

1. **Backbone model**: an EfficientNet image classifier (via the `timm` library), trained as a binary "normal vs. anomaly" classifier rather than the many-way galaxy classifier GalaxyMNIST was originally built for - we deliberately collapse the dataset down to a 1-vs-rest binary problem to match how AnomalyMatch is actually used (find the rare unusual class among a sea of normal-looking sources).
2. **FixMatch** (Sohn et al. 2020): a semi-supervised training algorithm. It uses the small labeled set directly, and additionally exploits the much larger *unlabeled* pool by training the model to agree with its own predictions on weakly vs. strongly augmented versions of the same unlabeled image (consistency regularization + pseudo-labeling). This is what lets the method work with only 5-20 labeled seed examples instead of thousands.
3. **Active learning loop**: after each training run, the current model scores every unlabeled image, the most likely mislabeled/ambiguous top-scoring examples are surfaced, their labels are corrected, and the model is retrained - repeated for several "training runs" (we use 2 here; the paper's headline GalaxyMNIST result uses 3).

The method has already been applied by its original authors to the Hubble Legacy Archive (99.6M cutouts), JWST (lens candidate discovery), and Euclid Q1 (jellyfish galaxy discovery) - **not yet to any Chandra X-ray data**, which is the actual novel contribution this project is building toward. This notebook is purely a mechanics check on a dataset the original paper already used, before we touch real X-ray cutouts.

## What data is being used: GalaxyMNIST

[GalaxyMNIST](https://github.com/mwalmsley/galaxy_mnist) is a small, clean public benchmark of 10,000 SDSS galaxy images (8,000 train / 2,000 test at time of release; we use all 10,000 as one pool here), labeled by volunteer morphology classification (Galaxy Zoo) into 4 balanced classes (~2,500 each):

- **smooth & round**
- **smooth & cigar-shaped**
- **edge-on disk**
- **unbarred spiral**

It is *not* Chandra data or even X-ray data - it's a standard, fast-to-download optical benchmark that the AnomalyMatch paper itself uses to report a reproducible headline number (~94% precision in the top 1% of anomaly-ranked images, after 3 active-learning cycles). We pick one class index as the "anomaly" and treat the other three as "normal", matching the paper's own benchmark protocol (`paper_scripts/paper_benchmark.py` in the AnomalyMatch repo, which we run directly below rather than reimplementing it).

In [ ]:
import subprocess
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout)

import torch
print("torch:", torch.__version__, "| CUDA available:", torch.cuda.is_available())
assert torch.cuda.is_available(), (
    "No GPU detected - set Accelerator to GPU T4 x2 (or P100) in the notebook "
    "settings panel and restart the session before continuing."
)

## 1. Clone AnomalyMatch and apply known bug fixes

The two fixes below aren't CPU-workarounds - they're real bugs found while first running this locally, present regardless of hardware:

- **`.safetensors` vs `.pth`**: `anomaly_match` >=1.3.1 switched checkpoint format from pickle-based `.pth` to `.safetensors` (a security fix, per the package's own changelog), but the `paper_scripts/` benchmark script predates that change and still hardcodes `.pth` filenames when copying saved checkpoints, causing a `FileNotFoundError` right after training finishes.
- **TurboJPEG fallback**: the benchmark script eagerly instantiates a `TurboJPEG` decoder at import time; if the native `libjpeg-turbo` library isn't preinstalled in the environment, this crashes before anything runs, even though the same file already has a PIL fallback for the actual decode call - it's only the top-level instantiation that needs a try/except.

We also keep the env-var overrides for prediction batch size and dataloader worker count (defaulting to the original GPU-tuned values, `1000` and `4`) purely as a safety valve in case this runs on a smaller-VRAM accelerator than expected.

In [ ]:
import os
from pathlib import Path

if not Path("AnomalyMatch").exists():
    !git clone --depth 1 https://github.com/esa/AnomalyMatch.git
%cd AnomalyMatch
!pip install -q . galaxy-datasets seaborn

In [ ]:
from pathlib import Path

pb_path = Path("paper_scripts/paper_benchmark.py")
pu_path = Path("paper_scripts/paper_utils.py")

# --- paper_benchmark.py ---
src = pb_path.read_text()

if "faulthandler.enable()" not in src:
    src = src.replace('"""\n\nimport os\n',
                       '"""\n\nimport faulthandler\nfaulthandler.enable()\n\nimport os\n', 1)

src = src.replace(
    'jpeg_decoder = TurboJPEG()\n'
    'USE_TURBOJPEG = True\n'
    'logger.info("Using TurboJPEG for faster image decoding")',
    'try:\n'
    '    jpeg_decoder = TurboJPEG()\n'
    '    USE_TURBOJPEG = True\n'
    '    logger.info("Using TurboJPEG for faster image decoding")\n'
    'except Exception as e:\n'
    '    jpeg_decoder = None\n'
    '    USE_TURBOJPEG = False\n'
    '    logger.warning(f"TurboJPEG unavailable ({e}); falling back to PIL")'
)

# anomaly_match >=1.3.1 saves checkpoints as .safetensors, not .pth
src = src.replace('f"model_iter{iteration + 1}.pth"', 'f"model_iter{iteration + 1}.safetensors"')
src = src.replace('f"model.pth"', 'f"model.safetensors"')
src = src.replace('f"model_iteration_{iteration}.pth"', 'f"model_iteration_{iteration}.safetensors"')

src = src.replace(
    'batch_size = 1000  # Lowered batch size to avoid 32-bit indexing error from PyTorch',
    'batch_size = int(os.environ.get("ANOMALYMATCH_PRED_BATCH_SIZE", "1000"))',
)
pb_path.write_text(src)

# --- paper_utils.py ---
src = pu_path.read_text()
if "import torch\n" not in src:
    src = src.replace("import pandas as pd\n", "import pandas as pd\nimport torch\n", 1)
src = src.replace(
    "cfg.num_workers = 4\n    cfg.pin_memory = True",
    'cfg.num_workers = int(os.environ.get("ANOMALYMATCH_NUM_WORKERS", "4"))\n'
    "    cfg.pin_memory = torch.cuda.is_available()",
)
pu_path.write_text(src)

print("Patches applied.")

## 2. Prepare the GalaxyMNIST dataset

Downloads the 10,000-image GalaxyMNIST set via the `galaxy-datasets` package and converts it into the HDF5 + label-CSV format `paper_benchmark.py` expects. Takes a couple of minutes.

In [ ]:
%cd paper_scripts
!python prepare_datasets.py --dataset galaxymnist --img_size 224

## 3. Run the benchmark: baseline -> 2 active-learning cycles

- `--n_samples 20 --anomaly_ratio 0.2`: seed with 20 labeled examples (4 anomaly / 16 normal) - deliberately small, matching the "5-10+ labeled examples" scale the project plan calls for, not the paper's larger benchmark runs.
- `--anomaly_classes 1`: GalaxyMNIST class index 1 is treated as the anomaly, everything else as normal.
- `--training_runs 2`: two active-learning cycles (paper's headline number uses 3; two is enough to see whether the loop is doing anything, cheaper to run).
- `--train_iterations 10`: FixMatch training iterations per cycle.
- On GPU this should take well under the 30+ minutes per cycle it took single-threaded on CPU.

In [ ]:
import os

# Kaggle's host RAM (not GPU VRAM) is the constraint here: the script's default
# prediction batch size (1000) with 4 DataLoader worker subprocesses was tuned
# for a workstation with more system RAM than a Kaggle session gets, and
# triggered "tried to allocate more memory than is available" kernel restarts.
# Lower both - this only affects prediction/dataloading throughput, not results.
os.environ["ANOMALYMATCH_PRED_BATCH_SIZE"] = "200"
os.environ["ANOMALYMATCH_NUM_WORKERS"] = "2"

In [ ]:
!python paper_benchmark.py \
  --dataset galaxymnist --anomaly_classes 1 \
  --n_samples 20 --anomaly_ratio 0.2 \
  --train_iterations 10 --training_runs 2 --n_mislabeled 5 \
  --size 224 --skip_mock_ui --seed 0

## 4. Results

`results_summary.csv` has one row per evaluation point (baseline = iteration 0, then one row per training cycle). Compare `final_auroc` / `final_auprc` and the top-1% precision columns against the paper's reported ~94% top-1% precision on GalaxyMNIST after 3 cycles - we're running 2 cycles with a much smaller seed set, so an improving trend matters more here than matching that number exactly.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# locate results_summary.csv by search rather than assuming cwd -
# robust to kernel restarts / cwd drift between cells 8 and here
candidates = sorted(Path("/kaggle/working").rglob("results_summary.csv"))
if not candidates:
    raise FileNotFoundError(
        "No results_summary.csv found under /kaggle/working - scroll cell 10's "
        "output to its end and check it actually finished (look for a "
        "'Results saved to ...' log line or a traceback after 'Training complete.')."
    )
results_csv = candidates[-1]
print("Using:", results_csv)
summary = pd.read_csv(results_csv)
display(summary)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
x = range(len(summary))

auroc_col = "auroc" if "auroc" in summary.columns else "final_auroc"
auprc_col = "auprc" if "auprc" in summary.columns else "final_auprc"
axes[0].plot(x, summary[auroc_col], marker="o", label="AUROC")
axes[0].plot(x, summary[auprc_col], marker="o", label="AUPRC")
axes[0].set_xlabel("evaluation point (0 = baseline)")
axes[0].set_ylabel("score")
axes[0].set_title("AUROC / AUPRC across AL cycles")
axes[0].legend()

top1_col = [c for c in summary.columns if "top_1" in c and "precision" in c]
if top1_col:
    axes[1].plot(x, summary[top1_col[0]], marker="o", color="tab:red")
    axes[1].axhline(94, color="gray", linestyle="--", label="paper's ~94% (3 cycles, larger seed set)")
    axes[1].set_xlabel("evaluation point (0 = baseline)")
    axes[1].set_ylabel("top-1% precision (%)")
    axes[1].set_title("Top-1% anomaly precision")
    axes[1].legend()

plt.tight_layout()
plt.savefig("galaxymnist_validation_summary.png", dpi=150)
plt.show()

## Next steps

If AUROC/AUPRC/top-1% precision all improve across cycles here, that's the pipeline-mechanics validation this notebook set out to get (Setup Step 1 in `image_anomaly_detection/PLAN.md`) - confirmed on GPU, without the CPU-specific crash. The next step is adapting this same training loop to real Chandra Source Catalog imaging cutouts (via the SIA endpoint `http://cda.cfa.harvard.edu/csc21siap/queryImages`, already verified separately) instead of GalaxyMNIST, with extended/diffuse X-ray morphology as the proposed target anomaly class - see the parent project's `PLAN.md` and `README.md` for that work.